 **Author:** Chistiakova Maria 241-2

 # **Dataset:**

Dataset can be found here:
https://www.kaggle.com/datasets/eduardopalmieri/nba-player-stats-season-2425

# **Functional Description:**

**Data Loading:** Load the NBA statistics dataset using Pandas.

**Preprocessing:** Clean and preprocess the data to ensure it is ready for analysis.

**Overview:** Analysis will compare players to identify the top performers based on key metrics analyze their strategies. This will also help to identify which characteristics and strategies in general have the biggest impact on the success. This can be useful for coaches, analysts, or managers.

**Detailed overview:**

1. Identify Top Players by GmSc (Game Score).

2. Analyze Key Metrics for Top Players (Points (PTS), Rebounds (TRB), Assists (AST), Field Goal Percentage (FG%), Turnovers (TOV)).

3. Comparing percentage of Wins/Loses.

4. Compare Contributions in Wins (Res).

**Hypothesis checking:**

Players with higher points per game (PTS) tend to have lower assist averages (AST), suggesting that players focusing on scoring contribute less to team play and overall ball distribution. Conversely, players with higher assist averages are likely to score fewer points, prioritizing teamwork and facilitating others’ scoring opportunities. If it is true, I assume that players who focus more on PTS, rather than on AST are more effective in achieving success.

To test this hypothesis:

 1. Analyze the correlation between PTS and AST across all players (to prove the first part of the hypothesis)

 2. Categorize players into “scorers” (high PTS, low AST) and “team contributors” (high AST, moderate PTS).

 3. Evaluate which approach is more effective in achieving team success (to prove the second part of the hypothesis)

 **Discussion:**

 Notes about the general aim, the information that can be seen from the data and hypothesis. Information about what kind of data I was working with, what insights I have found, results of the hypothesis and conclusion from the aim.

In [ ]:
# loading the NBA statistics dataset using Pandas

import pandas as pd
df = pd.read_csv("database_24_25 3.csv")

In [ ]:
# cleaning and preprocessing the data

data_cleaned = df.copy()
data_cleaned['GmSc_cleaned'] = (
    data_cleaned['GmSc']
    .replace('', float('nan'))
    .astype(float)
)
data_cleaned['PTS_cleaned'] = (
    data_cleaned['PTS']
    .replace('', float('nan'))
    .astype(float)
)
data_cleaned['TRB_cleaned'] = (
    data_cleaned['TRB']
    .replace('', float('nan'))
    .astype(float)
)
data_cleaned['AST_cleaned'] = (
    data_cleaned['AST']
    .replace('', float('nan'))
    .astype(float)
)
data_cleaned['FG%_cleaned'] = (
    data_cleaned['FG%']
    .replace('', float('nan'))
    .astype(float)
)
data_cleaned['TOV_cleaned'] = (
    data_cleaned['TOV']
    .replace('', float('nan'))
    .astype(float)
)
data_cleaned['Res_cleaned'] = (
    data_cleaned['Res']
    .apply(lambda x: True if x == 'W' else (False if x == 'L' else None))
)


In [ ]:
# as we have several games with the same players, we need to find the mean of each column (except non numerical) of repeted players

data_mean = (
    data_cleaned.groupby('Player')
    .agg({
        'GmSc_cleaned': 'mean',
        'PTS_cleaned': 'mean',
        'TRB_cleaned': 'mean',
        'AST_cleaned': 'mean',
        'FG%_cleaned': 'mean',
        'TOV_cleaned': 'mean',
        'Res_cleaned': 'mean'
    })
    .reset_index()
)

data_mean.rename(columns={'Res_cleaned': 'Win_Percentage'}, inplace=True)

data_mean = data_mean.round(2)
data_mean


,Player,GmSc_cleaned,PTS_cleaned,TRB_cleaned,AST_cleaned,FG%_cleaned,TOV_cleaned,Win_Percentage
0,A.J. Green,5.52,7.15,1.77,0.92,0.42,0.46,0.38
1,AJ Johnson,0.08,0.50,0.25,0.25,0.25,0.25,0.50
2,Aaron Gordon,14.50,15.43,6.71,3.14,0.52,1.29,0.57
3,Aaron Holiday,3.69,4.25,0.75,1.50,0.41,0.38,0.62
4,Aaron Nesmith,6.73,9.17,4.00,1.00,0.55,0.83,0.33
...,...,...,...,...,...,...,...,...
468,Zach Edey,9.92,11.14,6.93,0.79,0.60,1.64,0.57
469,Zach LaVine,15.75,21.58,5.25,4.08,0.49,2.92,0.50
470,Zeke Nnaji,1.24,1.62,0.88,0.38,0.29,0.12,0.62
471,Ziaire Williams,6.90,9.57,4.71,1.00,0.39,0.93,0.36


In [ ]:
# calculating basic statistics for the numerical fields, which I will use for further analysis

descriptive_stats = data_mean[['GmSc_cleaned', 'PTS_cleaned', 'TRB_cleaned', 'AST_cleaned', 'FG%_cleaned', 'TOV_cleaned']].describe()
print("Descriptive Statistics:")
print(descriptive_stats)

Descriptive Statistics:
       GmSc_cleaned  PTS_cleaned  TRB_cleaned  AST_cleaned  FG%_cleaned  \
count    473.000000   473.000000   473.000000   473.000000   473.000000   
mean       6.965687     8.699810     3.452622     2.017273     0.395243   
std        5.778836     7.208464     2.656126     2.007198     0.161982   
min       -1.700000     0.000000     0.000000     0.000000     0.000000   
25%        2.370000     3.000000     1.330000     0.570000     0.330000   
50%        6.110000     6.830000     2.920000     1.470000     0.420000   
75%       10.240000    12.600000     4.850000     2.780000     0.490000   
max       31.740000    31.380000    13.700000    11.700000     1.000000   

       TOV_cleaned  
count   473.000000  
mean      1.093658  
std       0.947730  
min       0.000000  
25%       0.400000  
50%       0.860000  
75%       1.570000  
max       4.790000  


In [ ]:
# 1. Identifying Top Players by GmSc (Game Score)
# GmSc is a comprehensive performance metric based on stats like points, rebounds, assists, and others.
# I will rank players based on their average GmSc across all games and select the top 10 players for further analysis.

# Firstly, we need to build a graph (GmSc and Players)

import plotly.express as px
# let's make a new dataset with only top 10 players, sorted from the highest game score to the lowest
data_top_10 = data_mean.sort_values(by=['GmSc_cleaned'])[len(data_mean.sort_values(by=['GmSc_cleaned']))-10:len(data_mean.sort_values(by=['GmSc_cleaned']))]
# then we need to build a graph with the first 10 players with the highest score and rank them
fig = px.bar(
    data_top_10,
    x='Player',
    y='GmSc_cleaned',
    title='Identifying Top Players by GmSc (Game Score)',
    orientation='v',
    template='plotly_white',
    labels={'GmSc_cleaned': 'Game Score', 'Player': 'Player'},
    color_discrete_sequence=['orange']
)


fig.update_layout(
    legend_title_text='Performance'
)

fig.show()

In [ ]:
print(data_top_10['Player'])

279               Kevin Durant
299               LeBron James
105               De'Aaron Fox
267         Karl-Anthony Towns
365             Paolo Banchero
220               Jayson Tatum
405    Shai Gilgeous-Alexander
152      Giannis Antetokounmpo
26               Anthony Davis
348               Nikola Jokić
Name: Player, dtype: object


From the graph, we got that Kevin Durant, LeBron James, De'Aaron Fox, Karl-Anthony Towns, Paolo Banchero,  Jayson Tatum,  Shai Gilgeous-Alexander, Giannis Antetokounmpo,  Anthony Davis and Nikola Jokić are the 10 Top Players by GmSc (Game Score). We found out who are the most consistently impactful players and further, we will analyse them.

In [ ]:
# 2. Analyzing Key Metrics for Top Players (Points (PTS), Rebounds (TRB), Assists (AST), Field Goal Percentage (FG%), Turnovers (TOV)).

# I will compare these key statistics:
# • Points (PTS) — player scoring performance.
# • Rebounds (TRB) — activity on the boards.
# • Assists (AST) — contribution to team play.
# • Field Goal Percentage (FG%) — shooting efficiency.
# • Turnovers (TOV) — frequency of mistakes.
# Next I will create visualizations (e.g., bar charts or box plots) to compare players.

import plotly.express as px

data_top_10 = data_mean.sort_values(by=['GmSc_cleaned'])[len(data_mean.sort_values(by=['GmSc_cleaned']))-10:len(data_mean.sort_values(by=['GmSc_cleaned']))]


data_melted = data_top_10.melt(
    id_vars=['Player'],
    value_vars=['PTS_cleaned', 'TRB_cleaned', 'AST_cleaned', 'FG%_cleaned', 'TOV_cleaned' ],
    var_name='Metric',
    value_name='Value'
)


fig = px.bar(
    data_melted,
    x='Player',
    y='Value',
    color='Metric',
    title='Analyzing Key Metrics for Top Players',
    template='plotly_white',
    labels={'Value': 'Performance', 'Player': 'Player', 'Metric': 'Metric'},
    barmode='group',
    color_discrete_sequence=['pink', 'blue', 'red', 'green', 'yellow']
)

fig.update_layout(
    legend_title_text='Metric'
)

fig.show()



In [ ]:
data_top_10

,Player,GmSc_cleaned,PTS_cleaned,TRB_cleaned,AST_cleaned,FG%_cleaned,TOV_cleaned,Win_Percentage
279,Kevin Durant,20.80,27.56,6.56,3.44,0.55,3.33,0.89
299,LeBron James,20.83,23.31,8.62,9.23,0.51,3.62,0.69
105,De'Aaron Fox,20.88,28.80,5.00,5.67,0.50,3.67,0.53
267,Karl-Anthony Towns,22.71,26.23,12.38,3.00,0.53,1.77,0.54
365,Paolo Banchero,23.02,29.00,8.80,5.60,0.46,2.20,0.60
220,Jayson Tatum,23.24,29.71,7.86,5.86,0.46,2.86,0.79
405,Shai Gilgeous-Alexander,23.83,28.50,5.29,6.29,0.51,2.57,0.79
152,Giannis Antetokounmpo,25.94,31.38,12.38,5.92,0.61,3.15,0.38
26,Anthony Davis,26.96,31.08,11.17,2.58,0.57,2.25,0.75
348,Nikola Jokić,31.74,29.70,13.70,11.70,0.57,4.10,0.70


### Comparing the average PTS,  AST,  TRB,  FG%  and TOV for the top players clearly shows which players excel in specific metrics.

# Comparing PTS, AST, TRB:

### Identifying Scorers vs. All-Around Players

Players with the high bars in the PTS (Points) category but low bars in AST (Assists) or TRB (Rebounds) focus primarily on scoring.

Which are: De'Aaron Fox (PTS 28.80, TRB 5.00, AST 5.67), Jayson Tatum (PTS 29.71, TRB	7.86, AST	5.86), Shai Gilgeous-Alexander (PTS 28.50, TRB 5.29, AST 6.29), Anthony Davis	(PTS 31.08,	TRB 11.17	AST 2.58).

These players are offensive specialists who contribute to their team mainly by putting points on the board.

### Identifying All-Around Players.

Players with balanced bars across PTS, AST, and TRB contribute in multiple areas. This is: LeBron James (PTS 23.31, AST 8.62, TRB	9.23)

These players are versatile, helping their teams by scoring, setting up teammates, and securing rebounds.

Players with high bars across PTS, AST, and TRB indicate  rare, elite players. These players are typically the centerpiece of their teams and serve as consistent contributors in multiple facets of the game. This is: Giannis Antetokounmpo (PTS 31.38, AST 12.38, TRB 5.92).

# Comparing PTS and FG% (Field Goal Percentage):

### Identifying Efficiency vs. Volume Scorers.

A high scorers with a low or medium FG% is a volume shooter, taking many attempts to achieve their point total: Paolo Banchero (PTS 29.00,	FG% 0.46), Jayson Tatum (PTS 29.71, FG% 0.46) and Kevin Durant (PTS 27.56	FG% 0.55)

# Comparing AST and TOV:

### Identifying risk-takers:

Risk-takers have high AST but also high TOV (Turnovers), showing a willingness to make difficult plays. This is: Nikola Jokić	(AST 11.70, TOV 4.10)
Teams may prefer risk-takers in fast-paced games and safe players in situations requiring ball security.

# Comparing TRB and TOV:

### Identifying specialists in other metrics.

High TRB (Rebounds) and low TOV (Turnovers) may point to a defensive or rebounding specialist, this is Karl-Anthony Towns (TRB 12.38, TOV 1.77)


# Conclusion:

By visualizing this data, we can identify strategy of players and teams can make informed decisions about player roles and areas for improvement.


In [ ]:
# 3. Comparing percentage of Wins/Loses.
# I will create a bar chart displaying the percentage of wins and losses for each player. The visualization allows to easily compare how often players win and lose in the context of their overall performance.

import plotly.express as px
import pandas as pd

data_top_10 = data_mean.sort_values(by=['GmSc_cleaned'])[len(data_mean.sort_values(by=['GmSc_cleaned']))-10:len(data_mean.sort_values(by=['GmSc_cleaned']))]


data_top_10['Loss_Percentage'] = 1 - data_top_10['Win_Percentage']


data_melted = data_top_10.melt(
    id_vars='Player',
    value_vars=['Win_Percentage', 'Loss_Percentage'],
    var_name='Result',
    value_name='Percentage'
)


data_melted['Percentage'] *= 100


fig = px.bar(
    data_melted,
    x='Player',
    y='Percentage',
    color='Result',
    title='Win and Loss Contributions per Player (Top 10)',
    text='Percentage',
    labels={'Percentage': 'Percentage (%)', 'Player': 'Player', 'Result': 'Game Result'},
    color_discrete_map={'Win_Percentage': 'green', 'Loss_Percentage': 'red'}
)


fig.update_traces(texttemplate='%{text:.1f}%', textposition='auto')

fig.update_layout(
    barmode='stack',
    template='plotly_white',
    legend_title='Result',
    xaxis_title='Player',
    yaxis_title='Percentage (%)'
)


fig.show()

The visualization provides a clear picture of which players have higher win and loss percentages, further helping to assess their contribution to the team’s overall results. Kevin Durant (89%), Jayson Tatum (79%) and Shai Gilgeous-Alexander (79%) have the highest percentage of wins. However it does not show their level of performance. So we need to compare contributions in Wins (Res).

In [ ]:
# 4. Comparing contributions in Wins (Res).
# I will compare average GmSc in games won (Res=W) vs. games lost (Res=L).
# Then I will analyze which players maintain their performance level regardless of the game outcome.


import plotly.express as px
data_top_10 = data_mean.sort_values(by=['GmSc_cleaned'])[len(data_mean.sort_values(by=['GmSc_cleaned']))-10:len(data_mean.sort_values(by=['GmSc_cleaned']))]


data_top_10['Avg_GmSc_Wins'] = data_top_10['Win_Percentage'] * data_top_10['GmSc_cleaned']
data_top_10['Avg_GmSc_Losses'] = (1 - data_top_10['Win_Percentage']) * data_top_10['GmSc_cleaned']


fig = px.scatter(
    data_top_10,
    x='Avg_GmSc_Wins',
    y='Avg_GmSc_Losses',
    text='Player',
    title='Consistency Analysis: GmSc in Wins vs. Losses',
    labels={
        'Avg_GmSc_Wins': 'Average GmSc in Wins',
        'Avg_GmSc_Losses': 'Average GmSc in Losses'
    },
    template='plotly_white'
)

fig.add_shape(
    type='line',
    x0=min(data_top_10['Avg_GmSc_Wins']),
    y0=min(data_top_10['Avg_GmSc_Losses']),
    x1=max(data_top_10['Avg_GmSc_Wins']),
    y1=max(data_top_10['Avg_GmSc_Losses']),
    line=dict(color='Red', dash='dot'),
    xref='x',
    yref='y'
)


fig.update_traces(textposition='top center')


fig.show()

If a point on the graph is close to the red diagonal line (which represents the idea of "consistency"), it means the player shows similar GmSc results in both wins and losses. These players maintain their level of performance regardless of the outcome of the game.

If a point significantly deviates from the line, it could indicate that the player only performs well or poorly depending on whether their team wins or loses. For example, if a player has high GmSc values in wins and low values in losses, it might suggest that their performance heavily depends on the team’s success or the game situation.

LeBron James and Paolo Banchero maintain their level of performance regardless of the outcome of the game. Which shows that they are more succesful.



# Hypothesis:

I will check is it true that players with higher points per game (PTS) tend to have lower assist averages (AST), suggesting that players focusing on scoring contribute less to team play and overall ball distribution. Conversely, players with higher assist averages are likely to score fewer points, prioritizing teamwork and facilitating others’ scoring opportunities. If it is true, my hypothesis is that players who focus more on PTS, rather than on AST are more effective in achieving success.

In [ ]:
# Determine if there is a statistically significant negative correlation between points per game (PTS) and assists (AST).

import numpy as np
import plotly.express as px
from scipy.stats import pearsonr


# Calculate the Pearson correlation coefficient

data_top_10 = data_mean.sort_values(by=['GmSc_cleaned'])[len(data_mean.sort_values(by=['GmSc_cleaned']))-10:len(data_mean.sort_values(by=['GmSc_cleaned']))]

pts = data_top_10['PTS_cleaned']
ast = data_top_10['AST_cleaned']

# Computing correlation coefficient and p-value
correlation, p_value = pearsonr(pts, ast)

# Displaying the correlation value
print(f"Pearson Correlation Coefficient (PTS vs AST): {correlation:.2f}")
print(f"P-Value: {p_value:.4f}")


fig = px.scatter(
    data_top_10,
    x='PTS_cleaned',
    y='AST_cleaned',
    title='Correlation Between PTS and AST',
    labels={'PTS_cleaned': 'Points Per Game (PTS)', 'AST_cleaned': 'Assists Per Game (AST)'},
    trendline="ols",  # Ordinary Least Squares. It's a statistical method used to find the best-fitting straight line through a set of data points.
    template='plotly_white'
)


fig.show()

Pearson Correlation Coefficient (PTS vs AST): -0.17
P-Value: 0.6387


1. Pearson Correlation Coefficient (r): -0.17

 • Negative Correlation:
 • The negative sign indicates an inverse relationship between PTS (Points Per Game) and AST (Assists Per Game). This suggests that players who score more tend to have slightly fewer assists on average.
 • Weak Correlation:
 • The value of it is close to 0, which means the relationship between PTS and AST is weak. Scoring and assisting are not strongly tied together for the players in this dataset, but still tied.

2. P-Value: 0.6387

 • The p-value is very high, indicating that the observed correlation is not statistically significant. This means:
 • There is no strong evidence to suggest a meaningful relationship between PTS and AST.

3. Scatter Plot Observation

 • Spread of Points:
 • The scatter plot shows a broad spread of data points without a clear linear trend.
 • Regression Line:
 • The regression line is almost flat, reinforcing the weak relationship.

Correlation showed that there is a weak connection between points per game (PTS) and assist averages (AST). So we need to research it in a more detailed way with other method.

We can categorize players into “scorers” (high PTS, low AST), “team contributors” (high AST, moderate PTS) and balanced and compare their impact on game outcomes (GmSc and win percentage) to evaluate which approach is more effective in achieving team success.



In [ ]:
# i will visualize connection between PTS and AST

import plotly.express as px
import pandas as pd

data_top_10 = data_mean.sort_values(by=['GmSc_cleaned'])[len(data_mean.sort_values(by=['GmSc_cleaned']))-10:len(data_mean.sort_values(by=['GmSc_cleaned']))]


data_melted = data_top_10[['Player', 'PTS_cleaned', 'AST_cleaned']].melt(
    id_vars='Player',
    value_vars=['PTS_cleaned', 'AST_cleaned'],
    var_name='Statistic',
    value_name='Value'
)


fig = px.bar(data_melted, x='Player', y='Value', color='Statistic',
             barmode='group',
             title='Top 10 Players: PTS vs AST',
             labels={'Value': 'Value', 'Player': 'Player', 'Statistic': 'Statistic'},
             color_discrete_map={'PTS_cleaned': 'blue', 'AST_cleaned': 'violet'}
)


fig.show()


In [ ]:
# Categorizing players into “scorers” (high PTS, low AST), “team contributors” (high AST, moderate PTS), balanced and other, which do not fit into any of the categories

import pandas as pd

data_top_10 = data_mean.sort_values(by=['GmSc_cleaned'])[len(data_mean.sort_values(by=['GmSc_cleaned']))-10:len(data_mean.sort_values(by=['GmSc_cleaned']))]

def categorize_player(row):
    if row['PTS_cleaned'] > 26 and row['AST_cleaned'] < 6:
        return 'Scorer'
    elif row['AST_cleaned'] > 6 and row['PTS_cleaned'] < 24:
        return 'Team Contributor'
    else:
        return 'Balanced'


data_top_10['Category'] = data_top_10.apply(categorize_player, axis=1)

print(data_top_10[['Player', 'PTS_cleaned', 'AST_cleaned', 'Category']])

                      Player  PTS_cleaned  AST_cleaned          Category
279             Kevin Durant        27.56         3.44            Scorer
299             LeBron James        23.31         9.23  Team Contributor
105             De'Aaron Fox        28.80         5.67            Scorer
267       Karl-Anthony Towns        26.23         3.00            Scorer
365           Paolo Banchero        29.00         5.60            Scorer
220             Jayson Tatum        29.71         5.86            Scorer
405  Shai Gilgeous-Alexander        28.50         6.29          Balanced
152    Giannis Antetokounmpo        31.38         5.92            Scorer
26             Anthony Davis        31.08         2.58            Scorer
348             Nikola Jokić        29.70        11.70          Balanced


In [ ]:
# Evaluating which approach is more effective in achieving team success.

import pandas as pd
import plotly.express as px


data_top_10 = data_mean.sort_values(by=['GmSc_cleaned'])[len(data_mean.sort_values(by=['GmSc_cleaned']))-10:len(data_mean.sort_values(by=['GmSc_cleaned']))]


data_top_10['Player_Type'] = data_top_10.apply(
    lambda row: 'Scorers' if row['PTS_cleaned'] > 26 and row['AST_cleaned'] < 6
    else 'Team Contributors' if row['AST_cleaned'] > 6 and row['PTS_cleaned'] < 24
    else 'Balanced',
    axis=1
)



player_type_summary = data_top_10.groupby('Player_Type')['GmSc_cleaned'].mean().reset_index()


fig = px.bar(
    player_type_summary,
    x='Player_Type',
    y='GmSc_cleaned',
    title='Effectiveness of Scorers vs Team Contributors',
    labels={'GmSc_cleaned': 'Average Game Score (GmSc)', 'Player_Type': 'Player Type'},
    text='GmSc_cleaned',
    template='plotly_white',
    color='Player_Type',
    color_discrete_map={'Scorers': 'red', 'Team Contributors': 'blue', 'Balanced': 'fuchsia'}
)


fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')

fig.show()

We see from the bar chart that players who focus more on PTS, rather than on AST are more effective in achieving success (23.36 contrary to 20.83). Players who balance AST and PTS are the most succesful ones.

#**Conclusion from the hypothesis:**

Hypothesis is that players with higher points per game (PTS) tend to have lower assist averages (AST), suggesting that players focusing on scoring contribute less to team play and overall ball distribution (conversely, players with higher assist averages are likely to score fewer points, prioritizing teamwork and facilitating others’ scoring opportunities) is proved. It is true. Connection is weak, but still exists.

Hypothesis is that players who focus more on PTS, rather than on AST are more effective in achieving success is proved. It is true.




#**Data transformation:**


First, I have cleaned 7 columns to get a new dataset data_cleaned.

Then I have created new data set data_mean with the same 7 cleaned same columns, but modified as we had several games with the same players. New columns mean of each column (except non numerical) of repeted players. For non numerical column I renamed 'Res_cleaned' to 'Win_Percentage' and counted quantity of wins divided by wins + loses.

Then I remained only rows with top 10 players sorted by game score to create a new data set data_top_10. Further I analysed only these 10 players.

#**Discussion:**

The general aim of the project is to compare players to identify the top performers based on key metrics and analyze their strategies. This will also help to identify which characteristics and strategies in general have the biggest impact on the success. This can be useful for coaches, analysts, or managers. This can be useful for coaches, analysts, or managers.

Firstly, I identified top 10 players according to average GmSc. GmSc is a comprehensive performance metric based on stats like points, rebounds, assists, and others. This is the best and fastest way to choose the most succesful players. After analysis I got that Kevin Durant, LeBron James, De'Aaron Fox, Karl-Anthony Towns, Paolo Banchero,  Jayson Tatum,  Shai Gilgeous-Alexander, Giannis Antetokounmpo,  Anthony Davis and Nikola Jokić are the 10 Top Players by average GmSc (Game Score). Highest score was 31.74 by Nikola Jokić.

Each player has its own strenght. To identify them, I decided to make more detailed analysis.

First, I decided to analyze Key Metrics for Top Players (Points (PTS) - player scoring performance, Rebounds (TRB) - activity on the boards, Assists (AST) - contribution to team play, Field Goal Percentage (FG%) - shooting efficiency, Turnovers (TOV)- frequency of mistakes) to group players according to their strengths. Comparing the average PTS,  AST,  TRB,  FG%  and TOV for the top players clearly shows which players excel in specific metrics.

Players with the high bars in the PTS (Points) category but low bars in AST (Assists) or TRB (Rebounds) focus primarily on scoring. These players are offensive specialists who contribute to their team mainly by putting points on the board: De'Aaron Fox (PTS 28.80, TRB 5.00, AST 5.67), Jayson Tatum (PTS 29.71, TRB	7.86, AST	5.86), Shai Gilgeous-Alexander (PTS 28.50, TRB 5.29, AST 6.29), Anthony Davis	(PTS 31.08,	TRB 11.17	AST 2.58).

Players with balanced bars across PTS, AST, and TRB contribute in multiple areas. These players are versatile, helping their teams by scoring, setting up teammates, and securing rebounds. This is: LeBron James (PTS 23.31, AST 8.62, TRB	9.23).

Players with high bars across PTS, AST, and TRB indicate  rare, elite players. These players are typically the centerpiece of their teams and serve as consistent contributors in multiple facets of the game. This is: Giannis Antetokounmpo (PTS 31.38, AST 12.38, TRB 5.92).

A high scorers with a low or medium FG% is a volume shooter, taking many attempts to achieve their point total: Paolo Banchero (PTS 29.00,	FG% 0.46), Jayson Tatum (PTS 29.71, FG% 0.46) and Kevin Durant (PTS 27.56	FG% 0.55).

Risk-takers have high AST but also high TOV (Turnovers), showing a willingness to make difficult plays. Teams may prefer risk-takers in fast-paced games and safe players in situations requiring ball security. This is: Nikola Jokić	(AST 11.70, TOV 4.10).

High TRB (Rebounds) and low TOV (Turnovers) may point to a defensive or rebounding specialist, this is Karl-Anthony Towns (TRB 12.38, TOV 1.77).

Then I compared percentage of Wins/Loses to further compare contributions in Wins (Res) and find out who maintain their level of performance regardless of the outcome of the game. First comparing showed that Kevin Durant (89%), Jayson Tatum (79%) and Shai Gilgeous-Alexander (79%) have the highest percentage of wins. However the second one showed that LeBron James and Paolo Banchero maintain their level of performance regardless of the outcome of the game. Which shows that they are more succesful.

Then I examined connection between PTS and AST. Correlation showed that there is a weak connection between points per game (PTS) and assist averages (AST). So I needed to research it in a more detailed way with other method.

I categorized players into “scorers” (high PTS, low AST) and “team contributors” (high AST, moderate PTS), balanced and other, which do not fit into any of categories and compared their impact on game outcomes (GmSc and win percentage) to evaluate which approach is more effective in achieving team success. I found out thet players who focus more on PTS, rather than on AST are more effective in achieving success (23.36 contrary to 20.83). Players who balance AST and PTS are the most succesful ones.







